# PromptTemplate

## 提示词模版

### 使用构造方法

In [ ]:
from langchain_core.prompts import PromptTemplate

# 生成格式化提示词模版
template = PromptTemplate(template="你是一个{role},请回答{question}",
                          input_variables=["role", "question"])
prompt = template.format(role="助手", question="你叫什么名字")
print(prompt)
print(type(prompt))

### from_template

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("你是一个{role},请回答{question}")
prompt = template.format(role="助手", question="你叫什么名字")
print(prompt)
print(type(prompt))


### 部分提示词模版

In [ ]:
from langchain_core.prompts import PromptTemplate

# partial_variables 固定部分变量
template = PromptTemplate.from_template("问题是{question}，答案是{answer}",
                                        partial_variables={"answer": "答案"})
prompt = template.format(question="你叫什么名字")
print(prompt)
print(type(prompt))


### 组合提示词

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("你是{role},解释{question}\n")
template2 = PromptTemplate.from_template("回答不超过{length}字")

template_combined = template + template2
prompt = template_combined.format(role="助手", question="你叫什么名字", length=10)
print(prompt)
print(type(prompt))

## 提示词方法

### partial

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("你是{role},解释{question}\n")
template_partial = template.partial(role="助手")
prompt = template_partial.format(question="你叫什么名字")
print(prompt)
print(type(prompt))

### invoke

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("你是{role},解释{question}\n")
prompt = template.invoke({"role": "助手", "question": "你叫什么名字"})
print(prompt)
print(type(prompt))  #StringPromptValue

print(prompt.to_string())  # StringPromptValue转换为str

# 对话提示词模版

## 聊天提示词模版

### 使用构造方法

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate([
    ("system", "你是一个助手，你的名字是{name}，请回答问题"),
    ("human", "你能干什么"),
    ("ai", "我可以做很多事情，比如回答问题，写代码，写文章，写书"),
    ("human", "解释{question}\n"),
])
prompt = template.format(name="小爱", question="langchain")
print(prompt)
print(type(prompt))

### from_messages

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "你是一个助手，你的名字是{name}，请回答问题"),
    ("human", "你能干什么"),
    ("ai", "我可以做很多事情，比如回答问题，写代码，写文章，写书"),
    ("human", "{question}是什么")
])
prompt = template.format(name="小爱", question="langchain")
print(prompt)
print(type(prompt))

### 结合LLM

In [ ]:
import os
from dotenv import load_dotenv
from langchain_community.chat_models import ChatTongyi
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate

load_dotenv(override=True)

template = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("你是{name}，请回答问题"),
    ("human", "你能干什么"),
    ("ai", "我可以做很多事情，比如回答问题，写代码，写文章，写书"),
    ("human", "{question}是什么")
])
message = template.format(name="小爱", question="langchain")

qwen_api_key = os.getenv("DASHSCOPE_API_KEY")

# 初始化 deepseek
model = ChatTongyi(
    model="qwen-plus",
    max_retries=2,
    api_key=qwen_api_key,
)

response = model.invoke(message)
print(response.content)


# FewShotPromptTemplate

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate

examples = [
    {"input": "小爱", "output": "小爱是一个助手"},
    {"input": "langchain", "output": "langchain是一个开源的库"},
]
template = FewShotPromptTemplate(
    examples=examples,  #人工示例
    example_prompt=PromptTemplate.from_template("输入：{input}\n输出：{output}"),
    prefix="请根据输入生成输出",
    suffix="输入：{input}\n输出：",  #用户真正的问题模版
    input_variables=["input"],  #suffix中需要的变量
    example_separator="\n",
)

prompt = template.format(input="langchain")
print(prompt)

# MessagePlaceHolder

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import MessagesPlaceholder, ChatPromptTemplate

# MessagesPlaceholder 占个坑，比如填入历史对话消息

template = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

prompt = template.format(input="你是谁，我又是谁",
                         history=[HumanMessage(content="你好,我是tmb"),
                                  SystemMessage(content="你是小爱")])
print(prompt)
qwen_api_key = os.getenv("DASHSCOPE_API_KEY")

# 初始化 deepseek
model = ChatTongyi(
    model="qwen-plus",
    max_retries=2,
    api_key=qwen_api_key,
)

response = model.invoke(prompt)
print(response.content)
